# **Uji Ketahanan Model terhadap Variasi Bahasa Lokal — Eksplorasi V5**

---

**Proyek:** KP Agentic AI + LLM  
**Tahap:** Pengujian eksploratif setelah audit V5  
**Dataset:** MASSIVE-Agents `id-ID` development

Pengujian membandingkan instruksi asli dan variasi V5 menggunakan empat model serta pemeriksa BFCL AST yang sama dengan pilot sebelumnya. Kandidat diambil dari hasil audit, bukan dari test set. Hasilnya digunakan untuk analisis eksploratif karena validasi manusia terhadap data belum selesai.

## **1. Konfigurasi Eksperimen**

---

Batch ini menggunakan 80 variasi, terdiri dari 20 instruksi untuk setiap kategori yang tersedia sebagai kandidat eksploratif. Kategori ID–Jawa tidak diikutkan karena masih memerlukan verifikasi penutur. Prompt dan pengaturan inferensi mengikuti pilot agar hasilnya dapat dibandingkan.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import sys
import time
import urllib.error
import urllib.request
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
from IPython.display import display

SEED_SAMPLE = 20261117
SEED_INFERENCE = 20261020
N_PER_CATEGORY = 20
RUN_NAME = "eksplorasi_v5_batch80"
API_BASE = "http://127.0.0.1:11434"
TIMEOUT_SECONDS = 1200
BFCL_VERSION = "2025.12.17"
PROMPT_VERSION = "compact_catalog_v1"

CATEGORIES = ["slang", "singkatan", "typo", "code_switch_id_en"]
MODELS = [
    "qwen2.5:7b-instruct",
    "gemma2:9b-instruct-q4_K_M",
    "Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m",
    "lfm2.5:8b-a1b-q4_K_M",
]
OPTIONS = {
    "temperature": 0,
    "seed": SEED_INFERENCE,
    "num_ctx": 16384,
    "num_predict": 384,
    "num_batch": 128,
}

print("Python:", sys.version.split()[0])
print("Kategori:", len(CATEGORIES))
print("Kandidat terpilih:", len(CATEGORIES) * N_PER_CATEGORY)
print("Jumlah prediksi variasi:", len(CATEGORIES) * N_PER_CATEGORY * len(MODELS))

Python: 3.11.9
Kategori: 4
Kandidat terpilih: 80
Jumlah prediksi variasi: 320


## **2. Dataset V5 dan Keputusan Audit**

---

Pemilihan kandidat menggunakan keputusan pada audit V5. Hanya data berstatus `KANDIDAT_EKSPLORASI_BELUM_HUMAN` yang masuk dalam populasi pengujian. Identitas instruksi, `group_id`, fungsi target, dan GT dicocokkan dengan sumber development agar variasi tetap dapat ditelusuri.

In [2]:
def locate_project():
    current = Path.cwd().resolve()
    candidates = [current, *current.parents, Path(r"D:\KP-Agentic-AI")]
    for root in candidates:
        if (root / "data" / "processed" / "massive_agents_reviewed" / "id-ID_development.jsonl").is_file():
            return root
    raise FileNotFoundError("Folder proyek dengan id-ID_development.jsonl tidak ditemukan.")


def read_csv(path):
    with Path(path).open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def read_jsonl(path):
    with Path(path).open("r", encoding="utf-8-sig") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def write_csv(path, rows, columns=None):
    if not rows and not columns:
        raise ValueError("Tidak ada baris atau kolom untuk disimpan.")
    columns = columns or list(rows[0])
    temp = path.with_suffix(".tmp")
    with temp.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    temp.replace(path)


def digest_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def as_bool(value):
    return value is True or str(value).strip().lower() == "true"


def compact_json(value):
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))


ROOT = locate_project()
DATA_DIR = ROOT / "data" / "processed" / "massive_agents_reviewed"
V5_DIR = ROOT / "results" / "variasi_bahasa_terkontrol" / "development_v5"
V5_FILE = V5_DIR / "kandidat_variasi_development_v5.csv"
AUDIT_FILE = V5_DIR / "KP_V5_AUDIT_KEPUTUSAN_DAN_PROTOKOL_EVALUASI.xlsx"
PILOT_CFG_FILE = ROOT / "results" / "ketahanan_bahasa_lokal" / "pilot_20_v1" / "config_pilot.json"

BASELINE_DIR = ROOT / "results" / "baseline_evaluation"
CFG_BASE_FILE = BASELINE_DIR / "config_development_full_v1_verified.json"
CFG_LFM_FILE = ROOT / "results" / "model_tambahan_lfm25" / "config_development_full.json"
AST_OLD_FILE = ROOT / "results" / "bfcl_ast_evaluation" / "ast_per_prediction.csv"
AST_LFM_FILE = ROOT / "results" / "model_tambahan_lfm25" / "ast_per_prediction_development_full.csv"

OUT = ROOT / "results" / "ketahanan_bahasa_lokal" / RUN_NAME

for path in [V5_FILE, AUDIT_FILE, PILOT_CFG_FILE, CFG_BASE_FILE, CFG_LFM_FILE, AST_OLD_FILE, AST_LFM_FILE]:
    if not path.is_file():
        raise FileNotFoundError(f"Input tidak ditemukan: {path}")

parent_list = read_jsonl(DATA_DIR / "id-ID_development.jsonl")
parents = {r["row_key"]: r for r in parent_list}
variants_v5 = read_csv(V5_FILE)
audited = pd.read_excel(AUDIT_FILE, sheet_name="KEPUTUSAN_971").fillna("")
audit_by_id = {str(r["variant_id"]): r for r in audited.to_dict("records")}

assert len(parent_list) == len(parents) == 397
assert len(variants_v5) == 971
assert len(audited) == len(audit_by_id) == 971
assert {v["variant_id"] for v in variants_v5} == set(audit_by_id)

eligible = []
for row in variants_v5:
    aid = audit_by_id[row["variant_id"]]
    if aid["keputusan_sementara"] != "KANDIDAT_EKSPLORASI_BELUM_HUMAN":
        continue
    parent = parents[row["row_key_induk"]]
    assert row["split"] == "development" and row["language_code"] == "id-ID"
    assert row["group_id"] == parent["group_id"] == str(aid["group_id"])
    assert row["instruksi_id_asli"] == parent["question"] == aid["instruksi_asli"]
    assert row["instruksi_variasi"] == aid["variasi_V5"]
    assert json.loads(row["gt_json"]) == parent["gt"] == json.loads(aid["gt_json"])
    assert row["fungsi_gt"] == next(iter(parent["gt"])) == aid["fungsi_GT"]
    assert row["instruksi_variasi"].strip() != parent["question"].strip()
    assert str(aid["validasi_manusia"]) in ("BELUM", "", "PENDING")
    assert str(aid["siap_evaluasi_final"]) == "TIDAK"
    eligible.append(row)

assert len(eligible) == 380, f"Populasi kandidat berubah: {len(eligible)}"
assert not any(r["kategori"] == "code_switch_id_jawa" for r in eligible)

pilot_config = json.loads(PILOT_CFG_FILE.read_text(encoding="utf-8-sig"))
pilot_ids = set(pilot_config["selected_variant_ids"])
assert len(pilot_ids) == 20

print("Kandidat V5:", len(variants_v5))
print("Kandidat eksploratif:", len(eligible))
print("Kategori:", dict(Counter(r["kategori"] for r in eligible)))
print("Variasi pilot yang dikecualikan:", len(pilot_ids & {r["variant_id"] for r in eligible}))

Kandidat V5: 971
Kandidat eksploratif: 380
Kategori: {'singkatan': 94, 'typo': 123, 'slang': 83, 'code_switch_id_en': 80}
Variasi pilot yang dikecualikan: 14


## **3. Pemilihan Sampel Berpasangan**

---

Sampel dipilih secara acak dengan seed tetap. Setiap `group_id` hanya muncul sekali pada batch ini, dan 20 variasi yang digunakan pada pilot tidak diulang. Perbandingan asli dan variasi dilakukan pada instruksi induk serta model yang sama.

In [3]:
rng = random.Random(SEED_SAMPLE)
selected = []
used_groups = set()

for category in CATEGORIES:
    candidates = sorted(
        (r for r in eligible if r["kategori"] == category and r["variant_id"] not in pilot_ids),
        key=lambda r: r["variant_id"],
    )
    rng.shuffle(candidates)
    chosen = []
    for row in candidates:
        if row["group_id"] in used_groups:
            continue
        chosen.append(row)
        used_groups.add(row["group_id"])
        if len(chosen) == N_PER_CATEGORY:
            break
    if len(chosen) != N_PER_CATEGORY:
        raise RuntimeError(f"Sampel kategori {category} hanya {len(chosen)} dari {N_PER_CATEGORY}")
    selected.extend(chosen)

assert len(selected) == 80
assert len({r["variant_id"] for r in selected}) == len(selected)
assert len(used_groups) == len(selected)
assert not (pilot_ids & {r["variant_id"] for r in selected})

print("Total variasi terpilih:", len(selected))
print("Komposisi:", dict(Counter(r["kategori"] for r in selected)))
print("Group unik:", len(used_groups))
display(pd.DataFrame(selected)[
    ["variant_id", "kategori", "instruksi_id_asli", "instruksi_variasi", "fungsi_gt"]
].head(10))

Total variasi terpilih: 80
Komposisi: {'slang': 20, 'singkatan': 20, 'typo': 20, 'code_switch_id_en': 20}
Group unik: 80


,variant_id,kategori,instruksi_id_asli,instruksi_variasi,fungsi_gt
0,id-ID:0001210::slang,slang,siapa yang saya temui hari ini,siapa yang aku temui hari ini,calendar.query
1,id-ID:0000657::slang,slang,hapus janji temu saya di tanggal tujuh maret,hapus janji temu aku di tanggal tujuh maret,calendar.remove
2,id-ID:0000920::slang,slang,saya ingin mendengarkan podcast terbaru mike a...,saya pengen mendengarkan podcast terbaru mike ...,play.podcasts
3,id-ID:0000029::slang,slang,bagaimana perkiraan cuaca hari ini di dki jakarta,gimana perkiraan cuaca hari ini di dki jakarta,weather.query
4,id-ID:0000396::slang,slang,bagaimana cuaca di probolinggo,gimana cuaca di probolinggo,weather.query
5,id-ID:0000366::slang,slang,saya ingin mendengarkan lagu sadis dari afgan,saya pengen mendengarkan lagu sadis dari afgan,play.music
6,id-ID:0000891::slang,slang,tolong tambahkan daftar barang yang harus dibe...,tolong dong tambahkan daftar barang yang harus...,lists.createoradd
7,id-ID:0000208::slang,slang,bagaimana cuaca besok,gimana cuaca besok,weather.query
8,id-ID:0001002::slang,slang,tolong pesankan saya tiket kereta ke surabaya,tolong dong pesankan saya tiket kereta ke sura...,transport.ticket
9,id-ID:0000248::slang,slang,tolong nyalakan mesin kopi untuk membuatkan sa...,tolong dong nyalakan mesin kopi untuk membuatk...,iot.coffee


## **4. Prompt dan Hasil Asli**

---

Katalog 55 fungsi dan pola prompt dipertahankan dari eksperimen baseline. Hasil AST dan pemilihan fungsi untuk instruksi asli diambil dari baseline development yang sudah tersedia, sehingga instruksi asli tidak perlu diuji ulang. Kecocokan versi prompt diperiksa melalui hash yang tersimpan pada konfigurasi baseline.

In [4]:
def compact_catalog(record):
    lines = []
    for func in record["function"]:
        description = func.get("description", "")
        description = description.split("The default user experience")[0]
        description = description.split(". ")[0].strip()[:100]
        properties = func.get("parameters", {}).get("properties", {})
        arguments = ", ".join(properties.keys())
        lines.append(f"{func['name']}: {description}\n  Arguments: {arguments if arguments else 'none'}")
    return "\n".join(lines)


def compact_prompt(record):
    return (
        "Select the most appropriate function for the user request.\n"
        "Return one JSON object with exactly two keys: name and arguments.\n"
        "The name must be a function from the catalog.\n"
        "Arguments must contain the relevant parameter names and string values.\n"
        "Do not copy function definitions or include explanations.\n\n"
        "Available functions:\n"
        + compact_catalog(record)
        + "\n\nUser request:\n"
        + record["question"]
        + '\n\nJSON output: {"name":"function_name","arguments":{}}'
    )


def calc_hashes(records):
    data_digest = hashlib.sha256()
    prompt_digest = hashlib.sha256()
    for record in records:
        data_digest.update(compact_json({
            "row_key": record["row_key"], "question": record["question"],
            "function": record["function"], "gt": record["gt"],
        }).encode("utf-8"))
        data_digest.update(b"\n")
        prompt_digest.update(compact_prompt(record).encode("utf-8"))
        prompt_digest.update(b"\n")
    return data_digest.hexdigest(), prompt_digest.hexdigest()


cfg_base = json.loads(CFG_BASE_FILE.read_text(encoding="utf-8-sig"))
cfg_lfm = json.loads(CFG_LFM_FILE.read_text(encoding="utf-8-sig"))

all_development = read_jsonl(DATA_DIR / "en-US_development.jsonl") + parent_list
all_ordered = sorted(all_development, key=lambda r: (str(r["source_record_id"]), r["language_code"]))
assert len(all_ordered) == 979
expected_data_sha, expected_prompt_sha = calc_hashes(all_ordered)
assert expected_data_sha == cfg_base["dataset_sha256"] == cfg_lfm["dataset_sha256"]
assert expected_prompt_sha == cfg_base["prompt_sha256"] == cfg_lfm["prompt_sha256"]

ast_formal_rows = read_csv(AST_OLD_FILE) + read_csv(AST_LFM_FILE)
formal_ast = {}
for row in ast_formal_rows:
    key = (row["model"], row["row_key"])
    if key in formal_ast:
        raise RuntimeError(f"AST baseline duplikat: {key}")
    formal_ast[key] = row

assert len(formal_ast) == 979 * len(MODELS)
assert all((model, row["row_key_induk"]) in formal_ast for model in MODELS for row in selected)

adapted = {}
for row in selected:
    record = dict(parents[row["row_key_induk"]])
    record["question"] = row["instruksi_variasi"]
    assert len(record["function"]) == 55
    adapted[row["variant_id"]] = record

print("Hash dataset dan prompt baseline: PASS")
print("Baseline AST:", len(formal_ast))
print("Instruksi asli diuji ulang: tidak")

Hash dataset dan prompt baseline: PASS
Baseline AST: 3916
Instruksi asli diuji ulang: tidak


## **5. Identitas Model dan BFCL**

---

Identitas empat model, digest, dan versi pemeriksa AST dicocokkan dengan pengujian sebelumnya. Hal ini menjaga agar perubahan skor tidak disebabkan oleh pergantian model atau evaluator. Pengaturan CPU khusus Gemma 2 dan Sahabat-AI dipertahankan sesuai pilot.

In [5]:
def ollama_request(endpoint, payload=None, timeout=TIMEOUT_SECONDS):
    data = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = urllib.request.Request(
        API_BASE + endpoint,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST" if data is not None else "GET",
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:500]
        raise RuntimeError(f"Ollama HTTP {exc.code}: {detail}") from exc


try:
    tags = ollama_request("/api/tags", timeout=20)
except Exception as exc:
    raise RuntimeError("Server Ollama lokal tidak dapat diakses.") from exc

installed = {m.get("name"): m for m in tags.get("models", [])}
missing = [model for model in MODELS if model not in installed]
if missing:
    raise RuntimeError(f"Model tidak ditemukan: {missing}")

expected_digests = dict(cfg_base["model_digests"])
expected_digests[MODELS[-1]] = cfg_lfm["model_digest"]
for model in MODELS:
    if installed[model].get("digest") != expected_digests[model]:
        raise RuntimeError(f"Digest model tidak sesuai baseline: {model}")

try:
    current_bfcl = version("bfcl-eval")
except PackageNotFoundError as exc:
    raise RuntimeError("Paket bfcl-eval tidak tersedia di kernel.") from exc
if current_bfcl != BFCL_VERSION:
    raise RuntimeError(f"Versi BFCL berubah: {current_bfcl} (acuan: {BFCL_VERSION})")

from bfcl_eval.eval_checker.ast_eval.ast_checker import ast_checker
from bfcl_eval.constants.enums import Language
from bfcl_eval.constants.model_config import MODEL_CONFIG_MAPPING

for model in MODELS:
    MODEL_CONFIG_MAPPING[model] = SimpleNamespace(underscore_to_dot=False)

checker_name = next(
    name for name, config in MODEL_CONFIG_MAPPING.items()
    if "_" not in name and not config.underscore_to_dot
)
print("Model dan digest:", len(MODELS), "PASS")
print("BFCL AST:", current_bfcl)

Model dan digest: 4 PASS
BFCL AST: 2025.12.17


## **6. Inferensi Variasi Bahasa V5**

---

Instruksi variasi diuji pada empat model dengan prompt dan opsi yang sama seperti baseline. Prediksi disimpan setelah setiap permintaan, disertai identitas eksperimen. Inferensi yang telah tersimpan tidak diulang apabila proses terhenti.

In [6]:
def build_request(model, record):
    content = compact_prompt(record)
    payload = {
        "model": model, "stream": False,
        "options": dict(OPTIONS), "keep_alive": "5m",
    }
    if model.startswith("Supa-AI/"):
        payload["prompt"] = (
            "<start_of_turn>user\n" + content
            + "\n<end_of_turn>\n<start_of_turn>model\n"
        )
        payload["raw"] = True
        payload["options"]["stop"] = ["<end_of_turn>", "<eos>"]
        endpoint = "/api/generate"
    else:
        payload["messages"] = [{"role": "user", "content": content}]
        payload["format"] = "json"
        endpoint = "/api/chat"

    if model in (
        "gemma2:9b-instruct-q4_K_M",
        "Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m",
    ):
        payload["options"]["num_gpu"] = 0
        payload["keep_alive"] = 0
    elif model == "lfm2.5:8b-a1b-q4_K_M":
        payload["keep_alive"] = 0
    return endpoint, payload


def parse_model_json(raw):
    try:
        obj = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return None, "invalid_json"
    if not isinstance(obj, dict):
        return None, "invalid_structure"
    if not isinstance(obj.get("name"), str) or not isinstance(obj.get("arguments"), dict):
        return None, "invalid_structure"
    return obj, "parsed"


identity = {
    "run": RUN_NAME,
    "source_v5_sha256": digest_file(V5_FILE),
    "audit_sha256": digest_file(AUDIT_FILE),
    "pilot_config_sha256": digest_file(PILOT_CFG_FILE),
    "source_id_sha256": digest_file(DATA_DIR / "id-ID_development.jsonl"),
    "selected_variant_ids": [r["variant_id"] for r in selected],
    "model_digests": {model: installed[model]["digest"] for model in MODELS},
    "models": MODELS,
    "baseline_prompt_sha256": expected_prompt_sha,
    "options": OPTIONS,
    "seed_sample": SEED_SAMPLE,
    "bfcl_eval": BFCL_VERSION,
    "validation": "eksploratif_belum_human",
}
fingerprint = hashlib.sha256(compact_json(identity).encode("utf-8")).hexdigest()[:20]

OUT.mkdir(parents=True, exist_ok=True)
CONFIG_FILE = OUT / "config_eksplorasi.json"
SELECTED_FILE = OUT / "selected_variants_v5.csv"
PRED_FILE = OUT / "predictions_variants_v5.csv"
AST_FILE = OUT / "ast_variants_v5.csv"
PAIRED_FILE = OUT / "paired_comparison_v5.csv"
SUMMARY_FILE = OUT / "summary_v5.csv"
META_FILE = OUT / "metadata_eksplorasi.json"

if CONFIG_FILE.is_file():
    previous = json.loads(CONFIG_FILE.read_text(encoding="utf-8-sig"))
    if previous.get("fingerprint") != fingerprint:
        raise RuntimeError("Konfigurasi berbeda dari percobaan yang sudah tersimpan. Hasil lama tidak diubah.")
else:
    if PRED_FILE.exists():
        raise RuntimeError("Prediksi ditemukan tanpa file konfigurasi. Hasil lama tidak diubah.")
    CONFIG_FILE.write_text(
        json.dumps({**identity, "fingerprint": fingerprint}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

write_csv(SELECTED_FILE, selected)
PRED_COLUMNS = [
    "fingerprint", "model", "variant_id", "row_key_induk", "group_id", "kategori",
    "instruksi_asli", "instruksi_variasi", "predicted_function",
    "prediction_arguments_json", "parsed_json", "done_reason", "prompt_tokens",
    "output_tokens", "latency_seconds", "raw_output", "timestamp_utc",
]

existing = read_csv(PRED_FILE) if PRED_FILE.is_file() else []
if existing and list(existing[0]) != PRED_COLUMNS:
    raise RuntimeError("Kolom prediksi yang sudah tersimpan berbeda.")
completed = set()
for row in existing:
    key = (row["model"], row["variant_id"])
    if key in completed or row["fingerprint"] != fingerprint:
        raise RuntimeError("Duplikasi atau fingerprint prediksi tidak sesuai.")
    completed.add(key)

print("Fingerprint:", fingerprint)
print("Prediksi tersimpan:", len(completed), "dari", len(selected) * len(MODELS))

for model in MODELS:
    print("MODEL:", model)
    for i, row in enumerate(selected, 1):
        key = (model, row["variant_id"])
        if key in completed:
            continue

        endpoint, payload = build_request(model, adapted[row["variant_id"]])
        started = time.perf_counter()
        response = ollama_request(endpoint, payload)
        raw = (
            response.get("response", "")
            if endpoint == "/api/generate"
            else (response.get("message") or {}).get("content", "")
        )
        prediction, parse_status = parse_model_json(raw)
        item = {
            "fingerprint": fingerprint,
            "model": model,
            "variant_id": row["variant_id"],
            "row_key_induk": row["row_key_induk"],
            "group_id": row["group_id"],
            "kategori": row["kategori"],
            "instruksi_asli": row["instruksi_id_asli"],
            "instruksi_variasi": row["instruksi_variasi"],
            "predicted_function": prediction["name"] if prediction else "",
            "prediction_arguments_json": compact_json(prediction["arguments"]) if prediction else "",
            "parsed_json": parse_status == "parsed",
            "done_reason": response.get("done_reason", ""),
            "prompt_tokens": response.get("prompt_eval_count", ""),
            "output_tokens": response.get("eval_count", ""),
            "latency_seconds": round(time.perf_counter() - started, 2),
            "raw_output": raw,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        }

        with PRED_FILE.open("a", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=PRED_COLUMNS)
            if handle.tell() == 0:
                writer.writeheader()
            writer.writerow(item)
        completed.add(key)
        existing.append(item)

        if i == 1 or i % 10 == 0 or i == len(selected):
            print(f"  {i}/{len(selected)} | {row['kategori']} | {parse_status}")

expected_count = len(MODELS) * len(selected)
if len(existing) != expected_count:
    raise RuntimeError(f"Prediksi belum lengkap: {len(existing)}/{expected_count}")
print("Total prediksi variasi:", len(existing))

Fingerprint: 68a866cb07368cb8d980
Prediksi tersimpan: 119 dari 320
MODEL: qwen2.5:7b-instruct
MODEL: gemma2:9b-instruct-q4_K_M
  40/80 | singkatan | parsed
  50/80 | typo | parsed
  60/80 | typo | parsed
  70/80 | code_switch_id_en | parsed
  80/80 | code_switch_id_en | parsed
MODEL: Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m
  1/80 | slang | parsed
  10/80 | slang | parsed
  20/80 | slang | parsed
  30/80 | singkatan | parsed
  40/80 | singkatan | parsed
  50/80 | typo | parsed
  60/80 | typo | parsed
  70/80 | code_switch_id_en | parsed
  80/80 | code_switch_id_en | parsed
MODEL: lfm2.5:8b-a1b-q4_K_M
  1/80 | slang | parsed
  10/80 | slang | parsed
  20/80 | slang | parsed
  30/80 | singkatan | parsed
  40/80 | singkatan | parsed
  50/80 | typo | parsed
  60/80 | typo | parsed
  70/80 | code_switch_id_en | parsed
  80/80 | code_switch_id_en | parsed
Total prediksi variasi: 320


## **7. Evaluasi BFCL AST dan Pemilihan Fungsi**

---

Prediksi dinilai menggunakan pemeriksa BFCL AST yang sama dengan pilot. AST menilai kecocokan fungsi dan argumen, sedangkan Function Selection Accuracy (FSA) hanya mengukur ketepatan fungsi target. GT tetap mengikuti data sumber dan tidak diubah.

In [7]:
paired = []
for pred in existing:
    parent = parents[pred["row_key_induk"]]
    target_fn = next(iter(parent["gt"]))
    formal = formal_ast[(pred["model"], pred["row_key_induk"])]
    parsed = as_bool(pred["parsed_json"])

    ast_ok = False
    error_type = ""
    error_detail = ""
    if parsed:
        args = json.loads(pred["prediction_arguments_json"])
        result = ast_checker(
            func_description=parent["function"],
            model_output=[{pred["predicted_function"]: args}],
            possible_answer=[parent["gt"]],
            language=Language.PYTHON,
            test_category="multiple_function",
            model_name=checker_name,
        )
        if not isinstance(result, dict) or not isinstance(result.get("valid"), bool):
            raise RuntimeError(f"Hasil AST tidak sesuai skema: {result!r}")
        ast_ok = result["valid"]
        if not ast_ok:
            error_type = str(result.get("error_type", "checker_error"))
            error_detail = json.dumps(result.get("error", []), ensure_ascii=False)
    else:
        error_type = "adapter:invalid_json"
        error_detail = "Prediksi tidak dapat diparse menjadi JSON fungsi"

    ast_original = as_bool(formal["bfcl_ast_correct"])
    fsa_original = formal["prediction_function"] == target_fn
    fsa_variant = pred["predicted_function"] == target_fn and parsed
    paired.append({
        "model": pred["model"],
        "variant_id": pred["variant_id"],
        "row_key_induk": pred["row_key_induk"],
        "group_id": pred["group_id"],
        "kategori": pred["kategori"],
        "fungsi_gt": target_fn,
        "fungsi_original": formal["prediction_function"],
        "fungsi_variasi": pred["predicted_function"],
        "fsa_original_correct": fsa_original,
        "fsa_variasi_correct": fsa_variant,
        "ast_original_correct": ast_original,
        "ast_variasi_correct": ast_ok,
        "ast_error_type": error_type,
        "ast_error_message": error_detail,
        "perubahan_ast": (
            "tetap_benar" if ast_original and ast_ok else
            "benar_ke_salah" if ast_original and not ast_ok else
            "salah_ke_benar" if not ast_original and ast_ok else "tetap_salah"
        ),
        "validasi_manusia": "BELUM",
        "status_penelitian": "EKSPLORATIF",
    })

assert len(paired) == len(existing) == 320
write_csv(AST_FILE, paired)
write_csv(PAIRED_FILE, paired)
print("AST asli benar:", sum(p["ast_original_correct"] for p in paired))
print("AST variasi benar:", sum(p["ast_variasi_correct"] for p in paired))
print("FSA asli benar:", sum(p["fsa_original_correct"] for p in paired))
print("FSA variasi benar:", sum(p["fsa_variasi_correct"] for p in paired))

AST asli benar: 45
AST variasi benar: 39
FSA asli benar: 230
FSA variasi benar: 212


## **8. Perbandingan Instruksi Asli dan Variasi**

---

Perbandingan dilakukan pada pasangan instruksi dan model yang sama. Perubahan AST dan FSA dihitung untuk setiap model dan kategori, termasuk jumlah prediksi yang berubah dari benar menjadi salah maupun sebaliknya.

In [8]:
summary = []
for model in MODELS:
    for category in [*CATEGORIES, "SEMUA"]:
        subset = [
            row for row in paired
            if row["model"] == model and (category == "SEMUA" or row["kategori"] == category)
        ]
        n = len(subset)
        count = lambda key: sum(bool(row[key]) for row in subset)
        ast_o, ast_v = count("ast_original_correct"), count("ast_variasi_correct")
        fsa_o, fsa_v = count("fsa_original_correct"), count("fsa_variasi_correct")
        summary.append({
            "model": model,
            "kategori": category,
            "n": n,
            "ast_asli_benar": ast_o,
            "ast_variasi_benar": ast_v,
            "ast_asli_pct": round(100 * ast_o / n, 2),
            "ast_variasi_pct": round(100 * ast_v / n, 2),
            "delta_ast_pp": round(100 * (ast_v - ast_o) / n, 2),
            "fsa_asli_pct": round(100 * fsa_o / n, 2),
            "fsa_variasi_pct": round(100 * fsa_v / n, 2),
            "delta_fsa_pp": round(100 * (fsa_v - fsa_o) / n, 2),
            "benar_ke_salah": sum(row["perubahan_ast"] == "benar_ke_salah" for row in subset),
            "salah_ke_benar": sum(row["perubahan_ast"] == "salah_ke_benar" for row in subset),
            "status": "EKSPLORATIF_BELUM_HUMAN",
        })
write_csv(SUMMARY_FILE, summary)

print("Ringkasan per model:")
display(pd.DataFrame(summary).query("kategori == 'SEMUA'")[
    ["model", "n", "ast_asli_pct", "ast_variasi_pct", "delta_ast_pp",
     "fsa_asli_pct", "fsa_variasi_pct", "delta_fsa_pp"]
])
print("Perubahan status AST:", dict(Counter(row["perubahan_ast"] for row in paired)))

Ringkasan per model:


,model,n,ast_asli_pct,ast_variasi_pct,delta_ast_pp,fsa_asli_pct,fsa_variasi_pct,delta_fsa_pp
4,qwen2.5:7b-instruct,80,21.25,18.75,-2.5,87.5,78.75,-8.75
9,gemma2:9b-instruct-q4_K_M,80,10.00,10.00,0.0,77.5,73.75,-3.75
14,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,80,18.75,18.75,0.0,80.0,80.00,0.00
19,lfm2.5:8b-a1b-q4_K_M,80,6.25,1.25,-5.0,42.5,32.50,-10.00


Perubahan status AST: {'tetap_salah': 267, 'tetap_benar': 31, 'salah_ke_benar': 8, 'benar_ke_salah': 14}


## **9. Pemeriksaan Perubahan Prediksi**

---

Kasus benar menjadi salah dan salah menjadi benar ditelusuri melalui nama fungsi yang dipilih dan galat AST. Hasil tetap bersifat deskriptif karena kesetaraan semantik variasi belum seluruhnya terverifikasi dan sampel tidak mencakup ID–Jawa.

In [9]:
changed = [row for row in paired if row["perubahan_ast"] in ("benar_ke_salah", "salah_ke_benar")]
details = []
variant_by_id = {row["variant_id"]: row for row in selected}
for row in changed:
    data = variant_by_id[row["variant_id"]]
    details.append({
        "model": row["model"],
        "kategori": row["kategori"],
        "perubahan_ast": row["perubahan_ast"],
        "instruksi_asli": data["instruksi_id_asli"],
        "instruksi_variasi": data["instruksi_variasi"],
        "fungsi_GT": row["fungsi_gt"],
        "fungsi_original": row["fungsi_original"],
        "fungsi_variasi": row["fungsi_variasi"],
        "error_ast_variasi": row["ast_error_type"],
    })

CHANGE_FILE = OUT / "perubahan_prediksi_v5.csv"
write_csv(CHANGE_FILE, details, columns=[
    "model", "kategori", "perubahan_ast", "instruksi_asli", "instruksi_variasi",
    "fungsi_GT", "fungsi_original", "fungsi_variasi", "error_ast_variasi"
])
print("Kasus status AST berubah:", len(details))
if details:
    display(pd.DataFrame(details).head(15))

Kasus status AST berubah: 22


,model,kategori,perubahan_ast,instruksi_asli,instruksi_variasi,fungsi_GT,fungsi_original,fungsi_variasi,error_ast_variasi
0,qwen2.5:7b-instruct,singkatan,salah_ke_benar,perlihatkan saya resep,perlihatkan sy resep,cooking.recipe,cooking.recipe,cooking.recipe,
1,qwen2.5:7b-instruct,singkatan,benar_ke_salah,bawa saya ke monas,bawa sy ke monas,transport.query,transport.query,transport.query,type_error:simple
2,qwen2.5:7b-instruct,typo,benar_ke_salah,definisikan terheran heran,dfeinisikan terheran heran,qa.definition,qa.definition,qa.definition,value_error:string
3,qwen2.5:7b-instruct,typo,benar_ke_salah,alarm apa saja yang sudah saya setel,aalrm apa saja yang sudah saya setel,alarm.query,alarm.query,alarm.query,type_error:simple
4,qwen2.5:7b-instruct,code_switch_id_en,benar_ke_salah,tolong putar lagu sayang,tolong play lagu sayang,play.music,play.music,play.music,value_error:string
5,qwen2.5:7b-instruct,code_switch_id_en,salah_ke_benar,mainkan slayer,play slayer,play.music,play.music,play.music,
6,gemma2:9b-instruct-q4_K_M,typo,benar_ke_salah,definisikan terheran heran,dfeinisikan terheran heran,qa.definition,qa.definition,qa.definition,value_error:string
7,gemma2:9b-instruct-q4_K_M,typo,benar_ke_salah,tampilkan waktu setempat di bandung saat ini,tmapilkan waktu setempat di bandung saat ini,datetime.query,datetime.query,datetime.query,type_error:simple
8,gemma2:9b-instruct-q4_K_M,typo,salah_ke_benar,mulai putar beberapa musik di radio,mulai putar beberapa msuik di radio,play.radio,play.radio,play.radio,
9,gemma2:9b-instruct-q4_K_M,code_switch_id_en,salah_ke_benar,mainkan slayer,play slayer,play.music,play.game,play.music,


## **10. Metadata dan Batasan Hasil**

---

Eksperimen ini menggunakan development set, 80 variasi, dan empat model. Hasil original diambil dari baseline yang sudah ada. Tidak ada pelatihan model, perubahan GT, ataupun penggunaan test set. Temuan belum digunakan sebagai klaim peningkatan kinerja final karena data masih memerlukan validasi manusia.

In [10]:
metadata = {
    "run": RUN_NAME,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "split": "development",
    "test_used": False,
    "GT_modified": False,
    "training_performed": False,
    "pilot_repeated": False,
    "models": MODELS,
    "total_variants": len(selected),
    "total_predictions": len(paired),
    "categories": dict(Counter(r["kategori"] for r in selected)),
    "unique_group_ids": len(used_groups),
    "audit_eligible_population": len(eligible),
    "pilot_excluded": len(pilot_ids & {r["variant_id"] for r in eligible}),
    "source_v5_sha256": digest_file(V5_FILE),
    "audit_sha256": digest_file(AUDIT_FILE),
    "baseline_prompt_sha256": expected_prompt_sha,
    "bfcl_version": current_bfcl,
    "bfcl_checker_config": checker_name,
    "model_digests": {model: installed[model]["digest"] for model in MODELS},
    "fingerprint": fingerprint,
    "human_validation": "PENDING",
    "approval_for_final_evaluation": False,
    "status": "development_exploratory_only",
    "notes": [
        "ID–Jawa tidak termasuk kandidat eksploratif terpilih.",
        "80 group_id berbeda digunakan untuk menghindari pengulangan induk pada batch ini.",
        "Tidak ada uji signifikansi yang diklaim pada data belum tervalidasi.",
        "Skor formal berasal dari baseline sebelumnya dan tidak diinferensi ulang.",
        "Jangan dibandingkan langsung dengan leaderboard atau skor paper tanpa protokol identik.",
    ],
}
META_FILE.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print("Eksperimen:", RUN_NAME)
print("Prediksi:", len(paired))
print("AST dan FSA:", SUMMARY_FILE.name)
print("Hasil berpasangan:", PAIRED_FILE.name)
print("Kasus perubahan:", CHANGE_FILE.name)
print("Metadata:", META_FILE.name)
print("Status: eksploratif, menunggu validasi manusia")

Eksperimen: eksplorasi_v5_batch80
Prediksi: 320
AST dan FSA: summary_v5.csv
Hasil berpasangan: paired_comparison_v5.csv
Kasus perubahan: perubahan_prediksi_v5.csv
Metadata: metadata_eksplorasi.json
Status: eksploratif, menunggu validasi manusia
